# Aim to test embedding models' performance for Traditional Chinese text
reference: [使用繁體中文評測各家 Embedding 模型的檢索能力](https://ihower.tw/blog/archives/12167)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
# from langchain_community.embeddings import SentenceTransformerEmbeddings
from datasets import load_dataset
from qdrant_client import QdrantClient
from langchain_qdrant import QdrantVectorStore
import pandas as pd
from langchain.document_loaders.dataframe import DataFrameLoader
import os

## Dataset Embedding and save to Qdrant

In [2]:
model_list = [
    'BAAI/bge-large-zh-v1.5',
    'TencentBAC/Conan-embedding-v1',
    'jinaai/jina-embeddings-v3',
    # 'intfloat/multilingual-e5-large',       # 微軟
    # 'infgrad/stella-large-zh-v2',           #
    # 'amazon/Titan-text-embeddings-v2',      # Amazon
    # ''
]

In [3]:
dataset_name = 'MediaTek-Research/TCEval-v2'
pure_name = 'TCEval-v2'
dataset = load_dataset('MediaTek-Research/TCEval-v2', 'drcd')

df = pd.DataFrame(dataset['test'])
df['references'] = df['references'].apply(lambda x: '|'.join(x))

loader = DataFrameLoader(df, page_content_column='paragraph')
docs = loader.load()

In [ ]:
for model_name in model_list:

    collection_name = model_name.split('/')[-1]
    if not os.path.exists(f"./db/MTK_{pure_name}/collection/{collection_name}"):
        embedding = HuggingFaceEmbeddings(
            model_name      = model_name,
            model_kwargs    = {'device': 'mps'}
        )
        
        QdrantVectorStore.from_documents(
            docs,
            path = f"./db/MTK_{pure_name}",
            collection_name = collection_name,
            embedding = embedding,
            force_recreate = True,
        )
        print(f'---  {model_name} done  ---')
        
    else:
        print(f'---  {model_name} already exists  ---')

## Load embedding model and make the comparison

In [ ]:
# vector_store = QdrantVectorStore(
#     client = QdrantClient(path = f"./db/MTK_{pure_name}"),
#     collection_name = model_list[0],
#     embedding = embedding,
# )
# retriever = vector_store.as_retriever(search_kwargs={'k': 5})

In [22]:
# raw1 = []
# raw2 = []
# for id, question in zip(df['id'], df['question']):
#     retrieval = retriever.invoke(question)
#     res_ids = [ d.metadata['id'] for d in retrieval ]
    
#     if id in res_ids:
#         hit_score = 1
#         mmr_score = 1/(res_ids.index(id)+1)
#     else:
#         hit_score = 0
#         mmr_score = 0
    
#     raw1.append(hit_score)
#     raw2.append(mmr_score)

In [ ]:
# res_df = pd.DataFrame(dataset['test'])
# res_df['hit_score'] = raw1
# res_df['mmr_score'] = raw2
# res_df.head()

In [ ]:
# res_df[['hit_score','mmr_score']].mean()